In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import networkx as nx
import copy

# ==============================
# PARÂMETROS GERAIS
# ==============================

L = 100
n_lagartos = L**2
estrategias = ['O', 'Y', 'B', 'E']

b = 2
c = 0.5

matriz_payoff = np.array([[1, c, b, 0],
                          [b, 1, c, 0],
                          [c, b, 1, 0],
                          [0, 0, 0, 0]])

index_map = {'O': 0, 'Y': 1, 'B': 2, 'E': 3}

n_geracoes = 50
n_pop = 1
sobreposicao = "s"
mi = 1
D = 0.1
sigma = 0.1

output_dir = f"C:/Unicamp/mestrado/simulacoes/main/RPS-diffusion/outputs/test/"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# ==============================
# CLASSE LAGARTO
# ==============================

class Lagarto:
    def __init__(self, i, j, estrategia, fitness,
                 coord_vizinhos, n_vizinhos):
        self.i = i
        self.j = j
        self.estrategia = estrategia
        self.fitness = 0.0
        self.coord_vizinhos = []
        self.n_vizinhos = n_vizinhos


    def calcular_coord_vizinhos(self, L):
        n = self.n_vizinhos
        if n <= 0:
            print("Erro: n_vizinhos <= 0")
            return

        i0, j0 = self.i, self.j

        def moore(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if not (dx == 0 and dy == 0)
            }

        def von_neumann(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if (dx != 0 or dy != 0) and (abs(dx) + abs(dy) <= r)
            }
        
        r = int(np.ceil((np.sqrt(1 + n) - 1) / 2))

        vn_r = von_neumann(r)
        mo_r = moore(r)

        if n <= 2 * r * (r + 1):    # faixa Moore(r-1) -> VN(r)
            base = moore(r - 1) if r > 1 else set()
            pool = list(vn_r - base)
        else:                       # faixa VN(r) -> Moore(r)
            base = vn_r
            pool = list(mo_r - base)

        faltam = n - len(base)
        if faltam <= 0:
            extras = []
        else:
            idx = np.random.choice(len(pool), size=faltam, replace=False)
            extras = [pool[k] for k in idx]

        self.coord_vizinhos = list(base) + extras

    def selection(self, G):
        vizinhos = vizinhos_unicos_rede(G, self)
        vizinho_escolhido = np.random.choice(vizinhos)

        if self.estrategia == 'O':
            if vizinho_escolhido.estrategia == 'Y':
                self.estrategia = 'E'
            elif vizinho_escolhido.estrategia == 'B':
                vizinho_escolhido.estrategia = 'E'
            elif vizinho_escolhido.estrategia == 'O':
                next
            elif vizinho_escolhido.estrategia == 'E':
                next
        elif self.estrategia == 'Y':
            if vizinho_escolhido.estrategia == 'B':
                self.estrategia = 'E'
            elif vizinho_escolhido.estrategia == 'O':
                vizinho_escolhido.estrategia = 'E'
            elif vizinho_escolhido.estrategia == 'Y':
                next
            elif vizinho_escolhido.estrategia == 'E':
                next
        elif self.estrategia == 'B':
            if vizinho_escolhido.estrategia == 'O':
                self.estrategia = 'E'
            elif vizinho_escolhido.estrategia == 'Y':
                vizinho_escolhido.estrategia = 'E'
            elif vizinho_escolhido.estrategia == 'B':
                next
            elif vizinho_escolhido.estrategia == 'E':
                next
        elif self.estrategia == 'E':
            next
    
    def reproduzir(self, G, mapa):
        vizinhos = vizinhos_unicos_rede(G, self)
        estrategias_vizinhos = [vizinho.estrategia for vizinho in vizinhos]


    def mortalidade(self, A, w):
        d = 1 / (1 + A * np.exp(w * self.fitness))
        a = np.random.rand()
        #print(f"Fitness: {self.fitness:.2f}, Mortalidade: {d:.4f}, Aleatório: {a:.4f}")
        if a < d:
            self.estrategia = 'E'
            #print("Lagarto morreu.")
        else:
            #print("Lagarto sobreviveu.")
            self.estrategia = self.estrategia

    def calcular_fitness_rede(self, G):
        fitness_total = 0.0
        for vizinho in vizinhos_unicos_rede(G, self):
            fitness_total += matriz_payoff[index_map[self.estrategia], index_map[vizinho.estrategia]]
        self.fitness = fitness_total

    def atualizar_links_lagarto(self, G, L, mapa):
        G.remove_edges_from(list(G.edges(self)))

        self.calcular_coord_vizinhos(L)
        #print(f"Vizinhos de ({self.estrategia}, {self.i}, {self.j}): {self.coord_vizinhos}")

        for (ni, nj) in self.coord_vizinhos:
            vizinho = mapa[(ni, nj)]
            G.add_edge(self, vizinho)

    def troca_posicao(self, G, m):
        vizinhos = vizinhos_unicos_rede(G, self)
        #print(f"Vizinhos de ({self.estrategia}): {[v.estrategia for v in vizinhos]}")
        vizinho = np.random.choice(vizinhos)
        #print(f"Vizinho escolhido para troca: ({vizinho.estrategia}, {vizinho.i}, {vizinho.j})")
        prob_migracao = m

        if np.random.rand() < prob_migracao:
            #print(f"Troca de posição entre ({self.estrategia}) e ({vizinho.estrategia})")
            self.estrategia, vizinho.estrategia = vizinho.estrategia, self.estrategia

    def ocupar_posicao(self, G):
        vizinhos = vizinhos_unicos_rede(G, self)
        #print(f"Vizinhos de ({self.estrategia}, {self.i}, {self.j}): {[v.estrategia for v in vizinhos]}")
        estrategias_vizinhos = [vizinho.estrategia for vizinho in vizinhos]
        opcoes = [e for e in estrategias_vizinhos if e != 'E']
        #print(f"Opções de ocupação para ({self.estrategia}, {self.i}, {self.j}): {opcoes}")
        if opcoes:
            self.estrategia = np.random.choice(opcoes)

# ==============================
# FUNÇÕES AUXILIARES
# ==============================

def vizinhos_unicos_rede(G, l):
    #return list(set(G.predecessors(no)).union(set(G.successors(no))))
    return list(G.neighbors(l))

def criar_lagartos():
    lista = []
    for i in range(L):
        for j in range(L):
            estrategia = np.random.choice(estrategias)
            lista.append(Lagarto(i, j, estrategia, 0, [], n_vizinhos = 8))
    return lista

def calcular_densidade_estrategia(G, estrategia):
    n = G.number_of_nodes()
    if n == 0:
        return 0.0

    contagem = sum(1 for lagarto in G.nodes if lagarto.estrategia == estrategia)
    return contagem / n

def calcular_densidade_rede(G):
    n = G.number_of_nodes()
    if n == 0:
        return 0.0
    ocupado = sum(1 for lagarto in G.nodes if lagarto.estrategia != 'E')
    return ocupado / n

def calcular_freq_estrategia_rede(G, estrategia):
    n = sum(1 for lagarto in G.nodes if lagarto.estrategia == 'O' or lagarto.estrategia == 'Y' or lagarto.estrategia == 'B')
    if n == 0:
        return 0.0

    contagem = sum(1 for lagarto in G.nodes if lagarto.estrategia == estrategia)
    return contagem / n

def grau_unico(G, l):
    #return len(set(G.predecessors(l)).union(set(G.successors(l))))
    return len(list(G.neighbors(l)))

def media_vizinhos_rede(G, lista_lagartos):
    graus = [grau_unico(G, l) for l in lista_lagartos]
    return np.mean(graus) if len(graus) > 0 else np.nan

def media_vizinhos_por_estrategia_rede(G, lista_lagartos):
    medias = []
    for e in estrategias:
        graus = [grau_unico(G, l) for l in lista_lagartos if l.estrategia == e]
        medias.append(np.mean(graus) if len(graus) > 0 else np.nan)
    return medias